In [32]:
import numpy as np
import pandas as pd
import os
import copy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from river.forest import ARFClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import warnings
warnings.filterwarnings("ignore")

def load_data(dataset_path='Dataset', csv_filename='ISCX_TOR_original.csv'):
    original_df = pd.read_csv(f'{dataset_path}/{csv_filename}')
    train_df, temp_df = train_test_split(original_df, test_size=0.4, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    return train_df, val_df, test_df

def train_base_model(train_df, val_df):
    X_train = train_df.iloc[:, :-1].values
    y_train = train_df.iloc[:, -1].values
    X_val = val_df.iloc[:, :-1].values
    y_val = val_df.iloc[:, -1].values
    
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    model = ARFClassifier(n_models=50, seed=42)
    
    for xi, yi in zip(X_train_scaled, y_train_encoded):
        model.learn_one({i: float(x) for i, x in enumerate(xi)}, yi)
    
    y_val_pred = []
    for xi in X_val_scaled:
        pred = model.predict_one({i: float(x) for i, x in enumerate(xi)})
        y_val_pred.append(pred if pred is not None else 0)
    
    val_accuracy = accuracy_score(y_val_encoded, y_val_pred)
    
    print("\n" + "="*60)
    print("BASE MODEL TRAINED ON ORIGINAL DATA")
    print("="*60)
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print(f"Number of classes: {len(label_encoder.classes_)}")
    
    return model, scaler, label_encoder

def load_perturbation_data(base_path='Dataset', attack_type='FGSM', test_perturb_levels=[0.1, 0.5, 1.0, 2.0, 5.0], scaler=None, label_encoder=None):
    datasets = []
    
    for perturb_level in sorted(test_perturb_levels):
        df = pd.read_csv(f'{base_path}/{attack_type}/{attack_type}_eps_{perturb_level}.csv')
        X = df.iloc[:, :-1].values
        y = df.iloc[:, -1].values
        
        y_encoded = label_encoder.transform(y)
        X_scaled = scaler.transform(X)
        
        datasets.append({
            'X': X_scaled,
            'y': y_encoded,
            'perturb_level': perturb_level
        })
    return datasets

def test_online_with_budget(base_model, X_test, y_test, perturb_level, update_budget=1.0):
    """
    update_budget: 0.0 to 1.0 (percentage of samples allowed for updating)
    - budget=1.0: updates on 100% of samples (full online learning)
    - budget=0.8: updates on 80% of samples, tests on 100%
    - budget=0.5: updates on 50% of samples, tests on 100%
    """
    model = copy.deepcopy(base_model)
    
    n_samples = len(X_test)
    n_updates_allowed = int(n_samples * update_budget)
    
    y_pred = []
    y_true_list = []
    updates_done = 0
    
    print(f"\n--- Level {perturb_level} (Budget: {update_budget*100}%) ---")
    
    for i, (xi, yi) in enumerate(zip(X_test, y_test)):
        xi_dict = {j: float(x) for j, x in enumerate(xi)}
        
        pred = model.predict_one(xi_dict)
        y_pred.append(pred if pred is not None else 0)
        y_true_list.append(yi)
        
        if updates_done < n_updates_allowed:
            model.learn_one(xi_dict, yi)
            updates_done += 1
        
        if (i + 1) % 500 == 0:
            current_acc = accuracy_score(y_true_list[-500:], y_pred[-500:])
            print(f"  Sample {i+1}: Updates={updates_done}, Recent Acc={current_acc:.4f}")
    
    accuracy = accuracy_score(y_true_list, y_pred)
    precision = precision_score(y_true_list, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true_list, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true_list, y_pred, average='weighted', zero_division=0)
    
    return {
        'Perturbation Level': perturb_level,
        'Update Budget %': update_budget * 100,
        'Total Samples': n_samples,
        'Updates Done': updates_done,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    }

def main(attack_type, test_perturb_levels):
    test_perturb_levels = test_perturb_levels
    attack_type = attack_type
    update_budget = 0.99  # 80% budget for updates (قابل تنظیم)
    
    train_df, val_df, test_df = load_data()
    
    base_model, scaler, label_encoder = train_base_model(train_df, val_df)
    
    perturb_datasets = load_perturbation_data(
        attack_type=attack_type, 
        test_perturb_levels=test_perturb_levels,
        scaler=scaler,
        label_encoder=label_encoder
    )
    
    print("\n" + "="*60)
    print(f"ONLINE TESTING WITH UPDATE BUDGET: {update_budget*100}%")
    print("Each level: Fresh base model → Predict → Update (up to budget limit)")
    print("="*60)
    
    results = []
    for dataset in perturb_datasets:
        result = test_online_with_budget(
            base_model, 
            dataset['X'], 
            dataset['y'], 
            dataset['perturb_level'],
            update_budget
        )
        results.append(result)
        
        print(f"\nLevel {result['Perturbation Level']} completed:")
        print(f"  Updates: {result['Updates Done']}/{result['Total Samples']}")
        print(f"  Accuracy: {result['Accuracy']:.4f}")
        print(f"  F1 Score: {result['F1 Score']:.4f}")
    
    df_results = pd.DataFrame(results)
    print("\n" + "="*60)
    print("FINAL RESULTS SUMMARY")
    print("="*60)
    print(df_results[['Perturbation Level', 'Updates Done', 'Accuracy', 'Precision', 'Recall', 'F1 Score']].to_string(index=False))
    
    if not os.path.exists('Results'):
        os.makedirs('Results')
    df_results.to_csv(f'Results/arf_budget_{int(update_budget*100)}_results.csv', index=False)
    print(f"\nResults saved to: Results/arf_budget_{int(update_budget*100)}_results.csv")

def test_multiple_budgets():
    """تابع تست برای بودجه‌های مختلف"""
    budgets = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    all_results = []
    
    train_df, val_df, test_df = load_data()
    base_model, scaler, label_encoder = train_base_model(train_df, val_df)
    
    perturb_datasets = load_perturbation_data(
        attack_type='FGSM',
        test_perturb_levels=[0.2, 0.5],
        scaler=scaler,
        label_encoder=label_encoder
    )
    
    for budget in budgets:
        print(f"\n{'='*60}")
        print(f"TESTING WITH BUDGET: {budget*100}%")
        print('='*60)
        
        for dataset in perturb_datasets:
            result = test_online_with_budget(
                base_model, 
                dataset['X'], 
                dataset['y'], 
                dataset['perturb_level'],
                budget
            )
            all_results.append(result)
            print(f"Level {result['Perturbation Level']}: Acc={result['Accuracy']:.4f}")
    
    df_all = pd.DataFrame(all_results)
    df_all.to_csv('Results/arf_all_budgets_results.csv', index=False)
    print("\nAll results saved to: Results/arf_all_budgets_results.csv")

if __name__ == "__main__":
    # سناریو 1: تست با بودجه مشخص (مثلاً 80%)
   
    pass
    # سناریو 2: تست با بودجه‌های مختلف (در صورت نیاز)
    # test_multiple_budgets()

In [27]:
if __name__ == "__main__":
    test_perturb_levels = [0.01, 0.05, 0.1, 0.2, 0.5]
    attack_type = 'PGD'
    main(attack_type, test_perturb_levels)


BASE MODEL TRAINED ON ORIGINAL DATA
Validation Accuracy: 0.9074
Number of classes: 8

ONLINE TESTING WITH UPDATE BUDGET: 20.0%
Each level: Fresh base model → Predict → Update (up to budget limit)

--- Level 0.01 (Budget: 20.0%) ---
  Sample 500: Updates=500, Recent Acc=0.9200
  Sample 1000: Updates=576, Recent Acc=0.9520
  Sample 1500: Updates=576, Recent Acc=0.9740
  Sample 2000: Updates=576, Recent Acc=0.9620
  Sample 2500: Updates=576, Recent Acc=0.9600

Level 0.01 completed:
  Updates: 576/2883
  Accuracy: 0.9525
  F1 Score: 0.9504

--- Level 0.05 (Budget: 20.0%) ---
  Sample 500: Updates=500, Recent Acc=0.9360
  Sample 1000: Updates=576, Recent Acc=0.9740
  Sample 1500: Updates=576, Recent Acc=0.9800
  Sample 2000: Updates=576, Recent Acc=0.9740
  Sample 2500: Updates=576, Recent Acc=0.9840

Level 0.05 completed:
  Updates: 576/2883
  Accuracy: 0.9705
  F1 Score: 0.9702

--- Level 0.1 (Budget: 20.0%) ---
  Sample 500: Updates=500, Recent Acc=0.9340
  Sample 1000: Updates=576, Rec

In [28]:
if __name__ == "__main__":
    test_perturb_levels = [0.1, 0.5, 1.0, 2.0, 5.0]
    attack_type = 'CW'
    main(attack_type, test_perturb_levels)


BASE MODEL TRAINED ON ORIGINAL DATA
Validation Accuracy: 0.9074
Number of classes: 8

ONLINE TESTING WITH UPDATE BUDGET: 20.0%
Each level: Fresh base model → Predict → Update (up to budget limit)

--- Level 0.1 (Budget: 20.0%) ---
  Sample 500: Updates=500, Recent Acc=0.8760
  Sample 1000: Updates=576, Recent Acc=0.9040
  Sample 1500: Updates=576, Recent Acc=0.9240
  Sample 2000: Updates=576, Recent Acc=0.9160
  Sample 2500: Updates=576, Recent Acc=0.9120

Level 0.1 completed:
  Updates: 576/2883
  Accuracy: 0.9032
  F1 Score: 0.8912

--- Level 0.5 (Budget: 20.0%) ---
  Sample 500: Updates=500, Recent Acc=0.8800
  Sample 1000: Updates=576, Recent Acc=0.9300
  Sample 1500: Updates=576, Recent Acc=0.9220
  Sample 2000: Updates=576, Recent Acc=0.9380
  Sample 2500: Updates=576, Recent Acc=0.9280

Level 0.5 completed:
  Updates: 576/2883
  Accuracy: 0.9185
  F1 Score: 0.9066

--- Level 1.0 (Budget: 20.0%) ---
  Sample 500: Updates=500, Recent Acc=0.8700
  Sample 1000: Updates=576, Recent 

In [30]:

if __name__ == "__main__":
    test_perturb_levels = [0.1, 0.5, 1.0, 2.0, 5.0]
    attack_type = "DeepFool" # or "FGSM" or "CW" or "DeepFool" or "RANDOM"
    main(attack_type, test_perturb_levels)


BASE MODEL TRAINED ON ORIGINAL DATA
Validation Accuracy: 0.9074
Number of classes: 8

ONLINE TESTING WITH UPDATE BUDGET: 20.0%
Each level: Fresh base model → Predict → Update (up to budget limit)

--- Level 0.1 (Budget: 20.0%) ---
  Sample 500: Updates=500, Recent Acc=0.8700
  Sample 1000: Updates=576, Recent Acc=0.8880
  Sample 1500: Updates=576, Recent Acc=0.8980
  Sample 2000: Updates=576, Recent Acc=0.8960
  Sample 2500: Updates=576, Recent Acc=0.8980

Level 0.1 completed:
  Updates: 576/2883
  Accuracy: 0.8890
  F1 Score: 0.8771

--- Level 0.5 (Budget: 20.0%) ---
  Sample 500: Updates=500, Recent Acc=0.8640
  Sample 1000: Updates=576, Recent Acc=0.8720
  Sample 1500: Updates=576, Recent Acc=0.8920
  Sample 2000: Updates=576, Recent Acc=0.8860
  Sample 2500: Updates=576, Recent Acc=0.8920

Level 0.5 completed:
  Updates: 576/2883
  Accuracy: 0.8783
  F1 Score: 0.8653

--- Level 1.0 (Budget: 20.0%) ---
  Sample 500: Updates=500, Recent Acc=0.8280
  Sample 1000: Updates=576, Recent 

In [33]:

if __name__ == "__main__":
    test_perturb_levels = [0.1, 0.5, 1.0, 2.0, 5.0]
    attack_type = "RANDOM" # or "FGSM" or "CW" or "DeepFool" or "RANDOM"
    main(attack_type, test_perturb_levels)


BASE MODEL TRAINED ON ORIGINAL DATA
Validation Accuracy: 0.9074
Number of classes: 8

ONLINE TESTING WITH UPDATE BUDGET: 99.0%
Each level: Fresh base model → Predict → Update (up to budget limit)

--- Level 0.1 (Budget: 99.0%) ---
  Sample 500: Updates=500, Recent Acc=0.8180
  Sample 1000: Updates=1000, Recent Acc=0.8060
  Sample 1500: Updates=1500, Recent Acc=0.8340
  Sample 2000: Updates=2000, Recent Acc=0.8400
  Sample 2500: Updates=2500, Recent Acc=0.8600

Level 0.1 completed:
  Updates: 2854/2883
  Accuracy: 0.8342
  F1 Score: 0.8091

--- Level 0.5 (Budget: 99.0%) ---
  Sample 500: Updates=500, Recent Acc=0.5940
  Sample 1000: Updates=1000, Recent Acc=0.6060
  Sample 1500: Updates=1500, Recent Acc=0.6040
  Sample 2000: Updates=2000, Recent Acc=0.6380
  Sample 2500: Updates=2500, Recent Acc=0.6540

Level 0.5 completed:
  Updates: 2854/2883
  Accuracy: 0.6282
  F1 Score: 0.5863

--- Level 1.0 (Budget: 99.0%) ---
  Sample 500: Updates=500, Recent Acc=0.4560
  Sample 1000: Updates=10